### Read the ERP product category data from the Bronze layer, apply the required cleaning and standardization steps, then save the final cleaned output to the Silver layer as a Delta table.

# Initialization



In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length

# Read Bronze table



In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

# Display the raw Bronze data first



In [0]:
# Display the raw Bronze data first to understand the file structure
# and identify the data quality issues that need cleaning in the Silver layer.
df.display()

- ============================================================

Data quality issues found in this file:
 1. Some string columns may contain leading or trailing spaces.
 2. The ID column should be checked because it will be used as the business key.
 3. The MAINTENANCE column should be standardized to consistent values.
 4. We should check for null or blank values in the main category columns.
 5. We should check for duplicate records based on the business key.
 6. Column names need to be renamed to cleaner business-friendly names before saving the final Silver table.

============================================================

# Silver Transformations



# Trimming

In [0]:
# Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Validation checks before cleaning



In [0]:
# Validation checks before cleaning ERP product category data
df.select(
    F.count(F.when(col("ID").isNull(), True)).alias("null_id_count"),
    F.count(F.when(trim(col("ID")) == "", True)).alias("blank_id_count"),
    F.count(F.when(col("CAT").isNull(), True)).alias("null_category_count"),
    F.count(F.when(trim(col("CAT")) == "", True)).alias("blank_category_count"),
    F.count(F.when(col("SUBCAT").isNull(), True)).alias("null_subcategory_count"),
    F.count(F.when(trim(col("SUBCAT")) == "", True)).alias("blank_subcategory_count"),
    F.count(F.when(col("MAINTENANCE").isNull(), True)).alias("null_maintenance_count"),
    F.count(F.when(~F.upper(col("MAINTENANCE")).isin("YES", "NO"), True)).alias("invalid_maintenance_value_count")
).display()

# Standardizing Maintenance



In [0]:
# Standardizing maintenance values
df = df.withColumn(
    "MAINTENANCE",
    F.when(F.upper(trim(col("MAINTENANCE"))) == "YES", "Yes")
     .when(F.upper(trim(col("MAINTENANCE"))) == "NO", "No")
     .otherwise("n/a")
)

# check duplicate record on bussiness keys in this file



In [0]:
# Check for duplicate records based on the business key
# to make sure the category rows are unique before saving to Silver.
df.groupBy("ID") \
  .count() \
  .filter(col("count") > 1) \
  .display()

### there is no dublicate

#rename coulmns



In [0]:
# Renaming columns to cleaner business names
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#quick validation for df



In [0]:
# Display the cleaned dataframe for a quick validation check
df.display()

#writing silver table


In [0]:
# Writing Silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

#check table created or not



In [0]:
%sql
-- Preview the final Silver table
SELECT * FROM workspace.silver.erp_product_category LIMIT 10